<a href="https://colab.research.google.com/github/JosephRini/embodied-ai-deployment/blob/main/notebooks/01_borrow_run_someone_elses_robot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 1 — Borrow: run someone else's robot

**The question this notebook answers:** what does a downloaded policy actually do, and can you trust the number it produces?

Today we will run **PushT**, which is a 2D simulated pushing task (dataset and small model) — a small circular agent pushes a T-shaped block to a target pose, observed through a single top-down camera. It's the simplest task in this course, which is exactly why it's the right one to start on: the physical world is only two numbers in, two numbers out, so nothing about the *task* gets in the way of understanding the *policy*.

**What you're doing today, concretely:**
1. Load the PushT dataset and look at one demonstration directly — not through a report about it.
2. Load `lerobot/diffusion_pusht`, a policy someone else trained, and run it.
3. Watch a success and a failure inline, in this notebook.
4. Compute the confidence interval yourself — the cell gives you the successes and n, you write the formula.
5. Write, in your own words, what the run could not show you. This is the one thing in this notebook nobody can do for you.

This notebook *replaces* the local n=50 PushT run — same result, same checkpoint, same task. It exists on GPU so a full run takes minutes instead of hours, and so every step is something you can see, not something you receive a report about.

Runtime: **Runtime → Change runtime type → T4 GPU**, then run cells top to bottom.

## A note on names — "PushT" means three different things here

Worth pinning down before going further, because the name gets reused:

- **`lerobot/pusht`** — the **dataset**. 206 human-teleop demonstrations, 25,650 frames. This is what a policy trains *on*. Used in §1 to look at a demonstration directly.
- **`lerobot/diffusion_pusht`** — the **model** (a Diffusion Policy checkpoint). Someone trained this on the dataset above and uploaded the resulting weights. This is what's loaded in §2 and evaluated in §3.
- **`PushT-v0`** (from `gym_pusht`) — the **environment**. A live simulator where the model's actions actually get executed and scored, step by step. Different from the dataset: the dataset is 206 recordings of a *human* doing the task; the environment is where the *policy* attempts it fresh, on a new random starting position each episode.

**"PushT"** on its own just names the *task* — push the T-shaped block to a target pose. The dataset, the model, and the environment are three separate objects that happen to share that word in their names.

Pipeline for §3's eval loop: environment gives an observation → model looks at it and outputs an action → environment steps forward and reports success or failure. Note that the dataset from §1 isn't used anywhere in this loop — it was only there so you could see what the training data looked like before running a model that had trained on it.

## 0. Setup
This cell is plumbing, not the lesson. Run it and move on.

In [2]:
!pip install -q lerobot==0.6.0 imageio[ffmpeg]
import torch, numpy as np
print('CUDA available:', torch.cuda.is_available())


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.1 MB/s eta 0:00:00
CUDA available: True


## 1. Look at the dataset directly
Before running any policy, look at one demonstration. This is a `LeRobotDataset` object — the same format
underlies every dataset you'll touch in this course (206 episodes of human teleop for PushT here).

In [4]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

dataset = LeRobotDataset('lerobot/pusht')
print('Number of episodes:', dataset.num_episodes)
print('Number of frames:', dataset.num_frames)

sample = dataset[0]
print('\nKeys in one sample:', list(sample.keys()))
for k, v in sample.items():
    if hasattr(v, 'shape'):
        print(f'  {k}: shape {tuple(v.shape)}')


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Number of episodes: 206
Number of frames: 25650

Keys in one sample: ['observation.image', 'observation.state', 'action', 'episode_index', 'frame_index', 'timestamp', 'next.reward', 'next.done', 'next.success', 'index', 'task_index', 'task']
  observation.image: shape (3, 96, 96)
  observation.state: shape (2,)
  action: shape (2,)
  episode_index: shape ()
  frame_index: shape ()
  timestamp: shape ()
  next.reward: shape ()
  next.done: shape ()
  next.success: shape ()
  index: shape ()
  task_index: shape ()


**What this above results shows us:**

- **206 human-teleop demonstrations, 25,650 total timesteps** — confirms this is the small, single-task dataset behind the "Diffusion Policy is not a foundation model" distinction.
- **`observation.state`: shape `(2,)`** — almost certainly the agent's (x, y) position.
- **`action`: shape `(2,)`** — almost certainly a target (x, y) to push toward.
- PushT's entire physical world is two numbers in, two numbers out.

**Before moving on, answer this yourself (edit this cell):**

- Which key holds the image? Which holds the action? a: image and action.
- What is the action's dimensionality? a:2
- What do those numbers physically mean for PushT? a: Whatever the agent controls, and for PushT that's almost certainly an (x, y) target position for the pusher to move toward, one number per spatial axis.


In [2]:
import matplotlib.pyplot as plt

img_key = [k for k in sample.keys() if 'image' in k][0]
img = sample[img_key]
img = img.permute(1, 2, 0).numpy() if img.shape[0] in (1, 3) else img.numpy()
plt.imshow(img)
plt.title(f'One frame from episode 0 ({img_key})')
plt.axis('off')
plt.show()


NameError: name 'sample' is not defined

## 2. Load the pretrained policy

`lerobot/diffusion_pusht` — trained by someone else, on this same dataset. You did not train this. Say that sentence out loud before running the next cell; it's the honest starting point for this whole notebook.

**What "pretrained" means here, concretely:**

- Someone ran a training job on the `lerobot/pusht` dataset from §1 — thousands of passes over those same 206 demonstrations — and uploaded the resulting weights to the Hugging Face Hub.
- `DiffusionPolicy.from_pretrained(...)` downloads two things: the **architecture config** (hyperparameters like `horizon`, `n_action_steps`, `num_inference_steps`) and the **trained weights** (`model.safetensors`) — then builds a live model with the config and loads the weights into it.
- Nothing in this cell computes a gradient or updates a weight. This is loading, not training.

**Why this distinction matters for the rest of the course:** everything in §3 tests this network's ability to generalize to *new* starting positions it never saw during training — that's what makes evaluation meaningful rather than circular. And it's the exact contrast the course keeps returning to: this notebook borrows a policy; Notebook 2 trains one from scratch; by Act 3 you'll be inheriting a much bigger pretrained model (SmolVLA) and hitting the limits of what "pretrained" can transfer.

**Before running the next cell, answer for yourself:** if this checkpoint has never seen your evaluation episodes before, what exactly is being measured when it succeeds or fails?

A: Whether the policy learned a generalizable pushing strategy from the 206
demonstrations, rather than memorizing those specific trajectories. Since
these episodes start from positions it never saw in training, success means
the skill transfers to new situations — not that it recalled something.

In [6]:
from lerobot.policies.diffusion.modeling_diffusion import DiffusionPolicy

policy = DiffusionPolicy.from_pretrained('lerobot/diffusion_pusht')
policy.to('cuda' if torch.cuda.is_available() else 'cpu')
policy.eval()
print(policy.config)


Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


[ERROR] `min_frames` is part of Qwen3VLVideoProcessorInitKwargs, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.12/dist-packages/transformers/models/qwen3_vl/video_processing_qwen3_vl.py.
[ERROR] `max_frames` is part of Qwen3VLVideoProcessorInitKwargs, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.12/dist-packages/transformers/models/qwen3_vl/video_processing_qwen3_vl.py.


config.json:   0%|          | 0.00/1.51k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.05GB            

model.safetensors: downloading bytes:           |  0.00B            

DiffusionConfig(n_obs_steps=2, input_features={'observation.image': PolicyFeature(type=<FeatureType.VISUAL: 'VISUAL'>, shape=(3, 96, 96)), 'observation.state': PolicyFeature(type=<FeatureType.STATE: 'STATE'>, shape=(2,))}, output_features={'action': PolicyFeature(type=<FeatureType.ACTION: 'ACTION'>, shape=(2,))}, device='cuda', use_amp=False, use_peft=False, push_to_hub=True, repo_id=None, private=None, tags=None, license=None, pretrained_path=None, pretrained_revision=None, horizon=16, n_action_steps=8, normalization_mapping={'ACTION': <NormalizationMode.MIN_MAX: 'MIN_MAX'>, 'STATE': <NormalizationMode.MIN_MAX: 'MIN_MAX'>, 'VISUAL': <NormalizationMode.MEAN_STD: 'MEAN_STD'>}, drop_n_last_frames=7, vision_backbone='resnet18', resize_shape=None, crop_ratio=1.0, crop_shape=(84, 84), crop_is_random=True, pretrained_backbone_weights=None, use_group_norm=True, spatial_softmax_num_keypoints=32, use_separate_rgb_encoder_per_camera=False, down_dims=(512, 1024, 2048), kernel_size=5, n_groups=8, 

**Answer yourself:**
- `num_inference_steps` — what does this control, and roughly how many forward passes does one action chunk cost?
A: NONE so it's default 100. Each action chunk costs 100 full denoising forward passes through the network before you get 16 actions out (8 of which get used).
- `n_action_steps` vs `horizon` — which one determines how often the policy is called again mid-episode?
A: the model predicts a chunk of 16 future actions at once (horizon), but only the first 8 get executed before it's called again with a fresh observation (n_action_steps). Answer is action steps.



**Sidenote bringing this all together:**

**How PushT works** (the task/environment): a small circular agent in a 2D world needs to push a T-shaped block until it lands in a target pose. That's it — physics, a shape, a goal position. It doesn't care what kind of policy is controlling the agent; a human playing with a mouse, a hand-coded rule, or any learned policy could attempt it.

**How the Diffusion Policy model works ** (the algorithm): denoising, 100 forward passes, action chunks of 16, execute 8, re-plan. That's a general-purpose recipe for turning "current camera image + current state" into "a sequence of actions," and it's completely agnostic to what task it's controlling. The exact same denoising mechanism trained on a different dataset would push a different shape, stack blocks, or steer a different robot — nothing about the 100-step denoising loop is PushT-specific.

The only place they touch is dimensionality: action: shape (2,) is small because PushT only needs a 2D target — that's PushT's fingerprint on the model's config, not the other way around. If this were a 6-DoF arm task, the same denoising mechanism would just be predicting 6 numbers per action instead of 2.

So: PushT is the problem; Diffusion Policy (with its 100-step denoising) is one possible solution to it, and it's a solution that would generalize to plenty of other problems with the right retraining.

## 3. Run the evaluation
n=50, matching your original local run, but on GPU this takes minutes. Each episode is an independent trial —
a new starting configuration — so this cell is *sampling*, not demonstrating.

this cell runs 50 independent trials of the policy attempting PushT, and records whether each one succeeded.

Concretely, for each of the 50 episodes it repeats the same loop:

1. Start fresh — reset the simulator to a new starting configuration (different each time, via the seed).
2. Look, decide, act, repeat — for up to 300 timesteps: the environment shows the policy an image + position → the policy (the 100-step denoising process from before) decides what to do → the environment executes that action and moves the simulation forward one step.
3.Check the outcome — stop early if the policy succeeds (T-block reached its target) or if the episode ends for some other reason (timeout, hard stop).
4. Record it — success or failure, plus every frame of that attempt (for the video you'll watch in §5).
5. Print a line and move to the next episode.

In [8]:
!pip install -q "lerobot[dataset]==0.6.0" gym-pusht "pymunk<6.5" imageio[ffmpeg]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.2 MB/s eta 0:00:00


In [1]:
import gymnasium as gym #gymnasium is the standard RL interface
import gym_pusht  # registers the PushT-v0 env
import imageio

N_EPISODES = 50
results = []       # (episode_idx, success: bool, frames: list of np arrays)

for ep in range(N_EPISODES):
    env = gym.make('gym_pusht/PushT-v0', obs_type='pixels_agent_pos') #asks the env to hand back both an image and the numeric agent position per step
    obs, _ = env.reset(seed=ep)
    frames = []
    success = False
    for step in range(300):
        frames.append(env.render())
        # NOTE: this is the minimal observation-to-tensor plumbing, not the lesson.
        # If it errors on a key name, that's real information -- read the error before asking for a fix.
        batch = {
            'observation.image': torch.from_numpy(obs['pixels']).permute(2, 0, 1).float().unsqueeze(0) / 255.0,
            'observation.state': torch.from_numpy(obs['agent_pos']).float().unsqueeze(0),
        }
        batch = {k: v.to(policy.device) for k, v in batch.items()}
        with torch.no_grad():
            action = policy.select_action(batch)
        obs, reward, terminated, truncated, info = env.step(action.squeeze(0).cpu().numpy())
        if info.get('is_success'):
            success = True
            break
        if terminated or truncated:
            break
    env.close()
    results.append({'episode': ep, 'success': success, 'frames': frames})
    print(f'episode {ep:2d}: {"success" if success else "failure"}')


AttributeError: 'Space' object has no attribute 'add_collision_handler'

## 4. Compute the confidence interval yourself
You have `results` — a list of 50 dicts with `success: bool`. Do not import a one-line binomial CI function and
call it a day. Write the Clopper-Pearson interval from `scipy.stats.beta`, by hand, so you know what it's doing.

Formula: for `k` successes out of `n`, the 95% CI is
`[beta.ppf(0.025, k, n-k+1), beta.ppf(0.975, k+1, n-k)]` (with edge cases at k=0 and k=n).

In [ ]:
from scipy.stats import beta

n = len(results)
k = sum(r['success'] for r in results)

# TODO (yours): implement the interval below. Replace the None values.
lower = None
upper = None

print(f'{k}/{n} = {k/n:.1%} success')
print(f'95% CI: [{lower}, {upper}]' if lower is not None else 'CI not yet computed -- fill in the cell above')


**Compare against your local n=50 run (78%, 95% CI [64.0%, 88.5%]).** Overlapping, roughly consistent,
or different? Given both are samples from the same frozen policy, what would explain a difference?
- *Your answer:*


## 5. Watch one success and one failure, inline
This is the cell that replaces the unwatched `eval_episode_1.mp4`. Pick one successful episode and one failed
episode from `results` and render them below.

In [ ]:
from IPython.display import Video, display

success_ep = next(r for r in results if r['success'])
failure_ep = next(r for r in results if not r['success'])

for label, ep in [('SUCCESS', success_ep), ('FAILURE', failure_ep)]:
    path = f'/content/episode_{label.lower()}.mp4'
    imageio.mimsave(path, ep['frames'], fps=10)
    print(f'--- {label} (episode {ep["episode"]}) ---')
    display(Video(path, embed=True, width=400))


## 6. What could this run not show you?

**This is the actual deliverable of this notebook. Nobody can fill it in for you — that's the point.**

Watch the failure video above at least twice. Each time you catch yourself asking a question you can't answer
from what's on screen, add a row: the question, what data would answer it, and whether that data exists
anywhere in this notebook's run right now.

Fill in the table below — add rows as you think of them, don't stop at three.

| Question | What data would answer it | Exists today? |
|---|---|---|
| *e.g. was the last action stale when the arm missed the T?* | *timestamp of chunk generation vs. timestamp of each executed action* | *no* |
| | | |
| | | |


## 7. Close the loop
One paragraph, in your own words: what does "78% success" actually mean now, after watching the failure and
writing the table above, versus what it meant as a bare number? This paragraph is Act 1's lesson —
write it here, then copy it into `writeups/lessons/eval.md` in the repo.

*Your paragraph:*
